# Differentiate

- Original source: [documentation:ref:differentiate](http://redberry.cc/documentation:ref:differentiate)
- Back to local index: [Redberry Documentation](../Redberry.Documentation.ipynb)


In [ ]:
#r "../../NRedberry.Core/bin/Debug/net10.0/NRedberry.Core.dll"
#r "../../NRedberry.Physics/bin/Debug/net10.0/NRedberry.Physics.dll"

using NRedberry;
using NRedberry.Tensors;
using TensorFactory = NRedberry.Tensors.Tensors;


---

### Description


- `Differentiate[var1, var2]` differentiates expression successively with respect to `var1`, `var2` ...


- `Differentiate` can differentiate tensorial expressions with respect to tensorial variables. `Differentiate` takes care about dummies relabelling and symmetries of tensors. The following convention is adopted:\[
 \frac{\partial T_{m_1 \dots m_k}}{\partial T_{n_1 \dots n_k}} = \delta^{n_1}_{m_1} \times \dots \times \delta^{n_k}_{m_k} \,+\, permutations
\]such that r.h.s. has exactly same symmetries as l.h.s.
### Examples

---

Derivative with respect to `x`:
```groovy
println Differentiate['x'] >> 'x**n'.t
```

```text
   > n*x**(n-1)
```

---

Derivative with respect to `f_mn`:
\[
\frac{\partial }{\partial f_{mn}} \, \left(\, f_{ab} f_{cd} \, \right) = \delta_a^m \, \delta_b^n \, f_{cd} \,+\, \delta_d^n \, \delta_c^m \, f_{ab}
\]

Redberry code:
```groovy
println Differentiate['f_mn'] >> 'f_ab*f_cd'.t
```

```text
   > d_a^m*d_b^n * f_cd + d_d^n*d_c^m*f_ab
```



---

Derivative involving a symbolic function `f_mn[t_ab]`:
\[
\frac{\partial }{\partial t_{mn}} \, \left(\, t_{ab} f^{ab}(t_{pq}) \, \right) =  f^{mn}(t_{dc}) \,+\, t_{ab} \frac{\partial }{\partial t_{mn}} \left( f^{ab}(t_{dc})\right)
\]

Redberry code:
```groovy
println Differentiate['t_mn'] >> 't_ab*f^ab[t_pq]'.t
```

```text
   > f^{mn}[t_{dc}] + t_{ab}*f~(1)^{abmn}[t_{dc}]
```


---

Derivative with respect to `x_m` and `y_m`:
\[
\frac{\partial^4 }{\partial x_{m} \partial x^m \partial y_n \partial y^n} \, \left(\, (x_a y^a)^5 \, \right) =  240\,  (y^b x_b)^3+40\, (y^b x_b)^3 \delta^a_a + 120\, (y_m y^m) (y^b x_b) (x_a x^a)
\]

Redberry code:
```groovy
def diff = Differentiate['x_m', 'x^m', 'y_a', 'y^a'] & CollectScalars
println diff >> '(x_a*y^a)**5'.t
```

```text
   > 240*(y^b*x_b)**3+40*(y^b*x_b)**3*d^a_a+120*y_m*y^m*y^b*x_b*x_a*x^a
```


---

Derivative with respect to antisymmetric tensor:
```groovy
setAntiSymmetric 'R_pq'
println Differentiate['R_mn'] >> 'R_ab'.t
```

```text
   > d_{a}^{m}*d_{b}^{n} - d_{a}^{n}*d_{b}^{m}
```

---


### Options

`Differentiate[var1, var2, ..., transformations]` allows to pass additional `transformations` which will be applied after each step of differentiation (for performance reasons):
```groovy
//setting up symmetries of Riemann tensor
addSymmetry 'R_abcd', -[1, 0, 2, 3].p
addSymmetry 'R_abcd', [2, 3, 0, 1].p

def tensor = 'R^acbd*Sin[R_abcd*R^abcd]'.t
def var1 = 'R^ma_m^b'.t,
    var2 = 'R^mc_m^d'.t
    
def diff1, diff2    
timing { 
  //take second derivative and then simplify
  diff1 = (Differentiate[var2, var1] & ExpandAndEliminate) >> tensor 
}
```

```text
   > Time: 1338 ms.
```

```groovy
timing { 
  //take second derivative and simplify permanently
  diff2 = Differentiate[var2, var1, ExpandAndEliminate] >> tensor
}
```

```text
   > Time: 14 ms.
```

```groovy
assert diff1 == diff2
```


### See also

- Related guides: [applying and manipulating transformations](../guide/ApplyingAndManipulatingTransformations.ipynb), representation of derivatives, [list of transformations](../guide/ListOfTransformations.ipynb)
- JavaDocs: [DifferentiateTransformation](http://api.redberry.cc/redberry/1.1.9/java-api/cc/redberry/core/transformations/DifferentiateTransformation.html)
- Source code: [DifferentiateTransformation.java](https://bitbucket.org/redberry/redberry/src/tip/core/src/main/java/cc/redberry/core/transformations/DifferentiateTransformation.java)
